In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('../data/ann_customer_churn_50000.csv')

In [3]:
df.head()

,age,tenure_months,monthly_charges,support_tickets,late_payments,usage_hours,satisfaction,contract_monthly,tech_support,auto_payment,churn
0,23,7,76.20,5,1,27.58,9.87,1,1,0,0
1,62,69,53.67,5,1,28.71,6.41,1,0,1,0
2,55,38,86.48,4,1,39.27,4.58,0,1,1,0
3,43,72,79.26,0,1,55.58,7.23,0,1,0,0
4,43,30,69.54,6,1,35.25,5.19,1,0,1,0


In [4]:
df.shape

(50000, 11)

In [5]:
df.isna().sum()

age                 0
tenure_months       0
monthly_charges     0
support_tickets     0
late_payments       0
usage_hours         0
satisfaction        0
contract_monthly    0
tech_support        0
auto_payment        0
churn               0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               50000 non-null  int64  
 1   tenure_months     50000 non-null  int64  
 2   monthly_charges   50000 non-null  float64
 3   support_tickets   50000 non-null  int64  
 4   late_payments     50000 non-null  int64  
 5   usage_hours       50000 non-null  float64
 6   satisfaction      50000 non-null  float64
 7   contract_monthly  50000 non-null  int64  
 8   tech_support      50000 non-null  int64  
 9   auto_payment      50000 non-null  int64  
 10  churn             50000 non-null  int64  
dtypes: float64(3), int64(8)
memory usage: 4.2 MB


<h2>Feature Selection</h2>

In [22]:
X=df.drop(columns=['churn'],axis=1)
y=df['churn']

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
from sklearn.preprocessing import StandardScaler

In [34]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)

In [36]:
X_test_scaled=scaler.transform(X_test)

In [37]:
y_train.to_numpy()

array([0, 0, 0, ..., 0, 0, 0], shape=(40000,))

In [38]:
y_test.to_numpy()

array([0, 0, 1, ..., 0, 0, 0], shape=(10000,))

<h3>Preparing data for pytorchby converting them to tensors</h3>

In [39]:
import torch

In [49]:
X_train_scaled_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy()).reshape(-1, 1)

X_test_scaled_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy()).reshape(-1, 1)

In [55]:
y_train_tensor = y_train_tensor.float()
y_test_tensor = y_test_tensor.float()

In [56]:
print(X_train_scaled_tensor.shape)
print(y_train_tensor.shape)

print(X_test_scaled_tensor.shape)
print(y_test_tensor.shape)

torch.Size([40000, 10])
torch.Size([40000, 1])
torch.Size([10000, 10])
torch.Size([10000, 1])


<h2>Neural Network Model 🧠</h2>

In [45]:
import torch.nn as nn

In [57]:
neural_model=nn.Sequential(
    nn.Linear(X_train_scaled_tensor.shape[1],16),
    nn.ReLU(),
    nn.Linear(16,8),
    nn.ReLU(),
    nn.Linear(8,1),
    nn.Sigmoid()
)

In [58]:
neural_model

Sequential(
  (0): Linear(in_features=10, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=1, bias=True)
  (5): Sigmoid()
)

In [66]:
loss_fn=nn.BCELoss()

In [74]:
optimizer=torch.optim.Adam(neural_model.parameters(),lr=0.001)

In [77]:
for epoch in range (1,101):
    optimizer.zero_grad()
    y_pred=neural_model(X_train_scaled_tensor)
    loss=loss_fn(y_pred,y_train_tensor)
    loss.backward()
    optimizer.step()

    print(f'{epoch}: {loss}')

1: 0.690456748008728
2: 0.6873409152030945
3: 0.684257447719574
4: 0.681204617023468
5: 0.6781824231147766
6: 0.6751886606216431
7: 0.6722239255905151
8: 0.6692873239517212
9: 0.6663766503334045
10: 0.6634905338287354
11: 0.6606287956237793
12: 0.657791256904602
13: 0.6549752950668335
14: 0.6521783471107483
15: 0.6494002938270569
16: 0.6466401219367981
17: 0.6438984274864197
18: 0.641171395778656
19: 0.638457715511322
20: 0.6357548832893372
21: 0.6330605745315552
22: 0.630374014377594
23: 0.6276945471763611
24: 0.6250203251838684
25: 0.6223493218421936
26: 0.6196812391281128
27: 0.6170142292976379
28: 0.6143468618392944
29: 0.6116770505905151
30: 0.6090030074119568
31: 0.6063227653503418
32: 0.6036348342895508
33: 0.6009378433227539
34: 0.5982300639152527
35: 0.5955113172531128
36: 0.5927814245223999
37: 0.590038537979126
38: 0.5872811675071716
39: 0.5845086574554443
40: 0.5817204117774963
41: 0.5789163112640381
42: 0.5760950446128845
43: 0.5732561349868774
44: 0.570400059223175
45: 0.